# Task 5 EDA v3 — behavior change and mechanical regularity

Test-only, unlabeled competition data. The lecturer exception for unsupervised Task 5 fitting is recorded in `docs/research-note-task5.md`; this notebook only constructs deterministic row-wise features. It writes exactly one `save_data` artifact. The datacard says `_1`–`_4` are consecutive periods but does not confirm which endpoint is newest, so endpoint and trend features are direction agnostic. Security flags are descriptive inputs, not labels.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_data.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.pipeline.random_seed import set_seed
SEED = set_seed()
import numpy as np
import pandas as pd
from IPython.display import display
from src.pipeline.save_data import save_data
SRC = ROOT / "src"
RAW = SRC / "task5" / "data" / "raw"
DATA_NAME = "eda_behavior_views_v3"
ID, TARGET = "player_id", "is_anomaly"
raw = pd.read_csv(RAW / "test.csv")
assert raw.shape == (25889, 123)
assert ID in raw and TARGET not in raw and raw[ID].is_unique
assert len(raw.columns.drop(ID)) == 122
assert all(pd.api.types.is_numeric_dtype(raw[c]) for c in raw.columns.drop(ID))
print("Unlabeled accounts:", len(raw), "raw predictors:", len(raw.columns)-1)

Unlabeled accounts: 25889 raw predictors: 122


In [2]:
# Match the v2 coordinate-quality rule before calculating movement.
base = raw.copy()
invalid = 0
for period in range(1, 5):
    lat, lon = f"login_lat_{period}", f"login_lon_{period}"
    bad = (base[lat].notna() & ~base[lat].between(-90, 90)) | (base[lon].notna() & ~base[lon].between(-180, 180))
    invalid += int(bad.sum())
    base.loc[bad, [lat, lon]] = np.nan
assert base[ID].equals(raw[ID]) and TARGET not in base
print("Invalid coordinate pairs masked:", invalid)

Invalid coordinate pairs masked: 265


In [3]:
FAMILIES = [
    "login_count", "location_changes", "device_count", "ip_hash_entropy",
    "login_time_variance", "session_duration_mean", "play_pattern_consistency",
    "activity_level", "skill_rating", "reaction_time_mean", "purchase_frequency",
    "purchase_amount_mean", "currency_transfers", "communication_frequency",
]
COHORT_FAMILIES = ["login_count", "activity_level", "purchase_frequency", "skill_rating"]
BOT_MEAN_FAMILIES = ["login_time_variance", "play_pattern_consistency", "session_duration_std",
                     "performance_stability", "reaction_time_std", "accuracy_std"]

def haversine_km(lat1, lon1, lat2, lon2):
    a1, b1, a2, b2 = [np.radians(x.astype(float)) for x in (lat1, lon1, lat2, lon2)]
    h = np.sin((a2-a1)/2)**2 + np.cos(a1)*np.cos(a2)*np.sin((b2-b1)/2)**2
    return 12742 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))

def add_behavior_views(frame):
    out = frame.copy()
    for family in FAMILIES:
        cols = [f"{family}_{i}" for i in range(1, 5)]
        values = out[cols].to_numpy(dtype=float)
        left = (out[cols[0]] - out[cols[1:]].median(axis=1)).abs()
        right = (out[cols[3]] - out[cols[:3]].median(axis=1)).abs()
        out[f"{family}_endpoint_surprise"] = pd.concat([left, right], axis=1).max(axis=1)
        steps = np.diff(values, axis=1)
        total = np.abs(steps).sum(axis=1)
        coherence = np.divide(np.abs(steps.sum(axis=1)), total, out=np.zeros(len(out)), where=total > 0)
        out[f"{family}_trend_coherence"] = np.where(np.isfinite(steps).all(axis=1), coherence, np.nan)
    for family in COHORT_FAMILIES:
        cols = [f"{family}_{i}" for i in range(1, 5)]
        out[f"{family}_median4"] = out[cols].median(axis=1)
    for family in BOT_MEAN_FAMILIES:
        cols = [f"{family}_{i}" for i in range(1, 5)]
        out[f"{family}_mean4"] = out[cols].mean(axis=1)
    for family in ["login_count", "activity_level", "session_duration_mean"]:
        cols = [f"{family}_{i}" for i in range(1, 5)]
        out[f"{family}_cv4"] = out[cols].std(axis=1) / (out[cols].mean(axis=1).abs() + 1)
    for i in range(1, 4):
        out[f"login_move_km_{i}{i+1}"] = haversine_km(
            out[f"login_lat_{i}"], out[f"login_lon_{i}"],
            out[f"login_lat_{i+1}"], out[f"login_lon_{i+1}"],
        )
    out["login_move_max_km"] = out[[f"login_move_km_{i}{i+1}" for i in range(1, 4)]].max(axis=1)
    return out

featured = add_behavior_views(base)
new_columns = [c for c in featured if c not in base]
assert len(new_columns) == 45, len(new_columns)
assert featured[base.columns].equals(base)
assert featured[ID].equals(raw[ID]) and TARGET not in featured
assert np.isfinite(featured.drop(columns=ID).to_numpy(dtype=float)[~featured.drop(columns=ID).isna().to_numpy()]).all()
print("New row-wise features:", len(new_columns), "total predictors:", len(featured.columns)-1)
print(new_columns)

New row-wise features: 45 total predictors: 167
['login_count_endpoint_surprise', 'login_count_trend_coherence', 'location_changes_endpoint_surprise', 'location_changes_trend_coherence', 'device_count_endpoint_surprise', 'device_count_trend_coherence', 'ip_hash_entropy_endpoint_surprise', 'ip_hash_entropy_trend_coherence', 'login_time_variance_endpoint_surprise', 'login_time_variance_trend_coherence', 'session_duration_mean_endpoint_surprise', 'session_duration_mean_trend_coherence', 'play_pattern_consistency_endpoint_surprise', 'play_pattern_consistency_trend_coherence', 'activity_level_endpoint_surprise', 'activity_level_trend_coherence', 'skill_rating_endpoint_surprise', 'skill_rating_trend_coherence', 'reaction_time_mean_endpoint_surprise', 'reaction_time_mean_trend_coherence', 'purchase_frequency_endpoint_surprise', 'purchase_frequency_trend_coherence', 'purchase_amount_mean_endpoint_surprise', 'purchase_amount_mean_trend_coherence', 'currency_transfers_endpoint_surprise', 'curren

In [4]:
# Descriptive EDA only: these flags are inputs, not ground truth.
summary = pd.DataFrame({"missing_pct": featured[new_columns].isna().mean()*100,
                        "median": featured[new_columns].median(),
                        "p99": featured[new_columns].quantile(.99)}).round(3)
display(summary)
flags = ["password_changed", "vpn_usage", "suspicious_login_time", "mass_item_sale"]
display(featured[flags].sum(axis=1).value_counts(dropna=False).sort_index().rename("accounts").to_frame())
for family in ["login_count", "location_changes", "device_count", "skill_rating", "currency_transfers"]:
    cols = [f"{family}_{i}" for i in range(1,5)]
    print(family, "adjacent Spearman 1→2:", round(featured[cols].corr(method="spearman").iloc[0,1],3),
          "endpoint surprise p99:", round(featured[f"{family}_endpoint_surprise"].quantile(.99),3))

,missing_pct,median,p99
login_count_endpoint_surprise,0.247,7.247,27.872
login_count_trend_coherence,17.961,0.223,1.000
location_changes_endpoint_surprise,0.267,0.339,10.011
location_changes_trend_coherence,18.131,0.360,1.000
device_count_endpoint_surprise,0.243,1.000,2.000
device_count_trend_coherence,17.853,0.200,1.000
ip_hash_entropy_endpoint_surprise,0.239,0.180,0.429
ip_hash_entropy_trend_coherence,18.378,0.320,1.000
login_time_variance_endpoint_surprise,0.228,0.755,2.676
login_time_variance_trend_coherence,18.301,0.335,1.000


,accounts
0.0,19302
1.0,5890
2.0,631
3.0,63
4.0,3


login_count adjacent Spearman 1→2: 0.886 endpoint surprise p99: 27.872
location_changes adjacent Spearman 1→2: 0.837 endpoint surprise p99: 10.011
device_count adjacent Spearman 1→2: 0.497 endpoint surprise p99: 2.0
skill_rating adjacent Spearman 1→2: 0.987 endpoint surprise p99: 1023.678


currency_transfers adjacent Spearman 1→2: 0.894 endpoint surprise p99: 19.3


In [5]:
# Save only this one named dataset. Existing artifacts are checked, never overwritten.
card_path = SRC / "task5" / "data" / "eda" / f"{DATA_NAME}.json"
if card_path.exists():
    card = json.loads(card_path.read_text(encoding="utf-8"))
    assert set(card["splits"]) == {"test"} and card["name"] == DATA_NAME
    old = pd.read_parquet(SRC / card["splits"]["test"]["path"])
    assert old.equals(featured), "Saved artifact differs; use a new name"
else:
    card = save_data(task=5, name=DATA_NAME, test=featured, format="parquet",
        card={"source": "task5/data/raw/test.csv", "feature_notebook": "task5/eda/build_eda_behavior_v3.ipynb",
              "preparation": "Masked invalid coordinate pairs; added 45 direction-agnostic row-wise change, trend, cohort, regularity, and movement features.",
              "feature_names": new_columns, "target": "unavailable; no offline F3",
              "period_order": "The datacard says consecutive periods; which endpoint is newest is unconfirmed.",
              "fitting": "None in EDA; peer and empirical percentile statistics belong in model fitting."})
print(card["splits"]["test"]["path"])

task5/data/eda/eda_behavior_views_v3-test.parquet


## EDA interpretation

All 25,889 accounts and 122 raw predictors are retained, with 265 invalid coordinate pairs masked. Four-period measurements often show account continuity: adjacent-period Spearman correlation is 0.886 for login count, 0.837 for location changes, 0.987 for skill rating, and 0.894 for currency transfers, but only 0.497 for device count. This supports testing change relative to an account's own history while warning that a one-step device change may be common. The 99th percentiles of endpoint surprise are 27.872 logins, 10.011 location changes, 2 devices, 1,023.678 skill-rating points, and 19.3 currency transfers. These are unlabeled distribution summaries, not anomaly thresholds or evidence of threat types. The period direction and duration remain undocumented; no impossible-travel speed is inferred.